# Movie Recommendation System

A content-based movie recommendation system built using the TMDB dataset.
It uses **TF-IDF Vectorization**, **Cosine Similarity**, and a **Hybrid Scoring** approach
that combines semantic similarity with real-world popularity and ratings.

---

## Pipeline Overview
1. Load & merge `movies.csv`, `credits.csv`, `poster.csv`
2. Feature Engineering (genres, keywords, top-3 cast, director)
3. TF-IDF Vectorization on combined tags
4. Cosine Similarity Matrix
5. Hybrid scoring: similarity + vote_average + popularity
6. Evaluation with Precision@K
7. Save model artifacts

## 1. Importing Libraries

In [13]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import warnings
warnings.filterwarnings('ignore')

## 2. Loading Data

In [14]:
movies  = pd.read_csv('Data/movies.csv')
credits = pd.read_csv('Data/credits.csv')
posters = pd.read_csv('Data/poster.csv')

print(f"Movies  : {movies.shape}")
print(f"Credits : {credits.shape}")
print(f"Posters : {posters.shape}")

Movies  : (4803, 20)
Credits : (4803, 4)
Posters : (9837, 2)


## 3. Merging and Cleaning

We merge all three datasets on `title` and retain the columns needed for feature engineering.
We also keep `vote_average` and `popularity` for use in hybrid scoring later.

In [15]:
movies = movies.merge(credits, on='title')
movies = movies.merge(posters,  on='title')

# Keep both semantic features AND numeric scores
movies = movies[['movie_id', 'title', 'overview',
                 'genres', 'keywords', 'cast', 'crew',
                 'vote_average', 'popularity', 'poster']]

movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

print(f"Total usable movies: {len(movies)}")
movies.head(2)

Total usable movies: 3028


,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity,poster
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,150.437577,https://image.tmdb.org/t/p/original/jRXYjXNq0C...
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,139.082615,https://image.tmdb.org/t/p/original/2YMnBRh8F6...


## 4. Feature Engineering

We extract:
- **Genres** and **Keywords** – all names from JSON list
- **Cast** – only the **top 3** actors (to reduce noise)
- **Crew** – only the **Director**

Multi-word names (e.g. `Sam Worthington`) are joined (`SamWorthington`) so they are treated as a single token during vectorization.

In [16]:
def extract_names(obj):
    """Extract 'name' field from every item in a JSON list."""
    return [i['name'].replace(" ", "") for i in ast.literal_eval(obj)]

def top3_cast(obj):
    """Extract top‑3 cast names."""
    return [i['name'].replace(" ", "") for i in ast.literal_eval(obj)[:3]]

def fetch_director(obj):
    """Return the director's name (empty list if not found)."""
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            return [i['name'].replace(" ", "")]
    return []

movies['genres']   = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast']     = movies['cast'].apply(top3_cast)
movies['crew']     = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.lower().split())

print("Feature extraction complete.")

Feature extraction complete.


In [17]:
# Build combined 'tags' column
movies['tags'] = (movies['overview'] + movies['genres'] +
                  movies['keywords'] + movies['cast'] + movies['crew'])

new_df = movies[['movie_id', 'title', 'tags',
                 'vote_average', 'popularity', 'poster']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

new_df.head(3)

,movie_id,title,tags,vote_average,popularity,poster
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...",7.2,150.437577,https://image.tmdb.org/t/p/original/jRXYjXNq0C...
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...",6.9,139.082615,https://image.tmdb.org/t/p/original/2YMnBRh8F6...
2,206647,Spectre,a cryptic message from bond’s past sends him o...,6.3,107.376788,https://image.tmdb.org/t/p/original/zj8ongFhtW...


## 5. TF-IDF Vectorization

Unlike a basic `CountVectorizer`, **TF-IDF** down-weights very common words and
up-weights terms that are distinctive to a small number of movies.  
This leads to significantly better similarity results.

In [18]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(new_df['tags'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (3028, 5000)


In [19]:
# Cosine Similarity
similarity = cosine_similarity(tfidf_matrix)

print(f"Similarity matrix shape: {similarity.shape}")

Similarity matrix shape: (3028, 3028)


## 6. Normalizing Signals for Hybrid Scoring

We normalize `vote_average` and `popularity` to a [0, 1] range so they
can be combined with the cosine similarity score on an equal footing.

**Hybrid Score Formula:**
```
score = similarity * 0.70 + vote_average_norm * 0.20 + popularity_norm * 0.10
```

In [20]:
# Min-Max normalize numeric signals
new_df['vote_norm'] = (
    (new_df['vote_average'] - new_df['vote_average'].min()) /
    (new_df['vote_average'].max() - new_df['vote_average'].min())
)
new_df['pop_norm'] = (
    (new_df['popularity'] - new_df['popularity'].min()) /
    (new_df['popularity'].max() - new_df['popularity'].min())
)

vote_arr = new_df['vote_norm'].values
pop_arr  = new_df['pop_norm'].values

print("Normalization complete.")

Normalization complete.


## 7. Recommendation Function

The `recommend()` function supports two modes:
- **`hybrid=True`** (default) – weighted combination of similarity, rating, popularity
- **`hybrid=False`** – pure cosine-similarity ranking

In [21]:
def recommend(movie, top_n=5, hybrid=True):
    """
    Return top‑N movie recommendations for a given title.

    Parameters
    ----------
    movie  : str   – exact movie title
    top_n  : int   – number of results (default 5)
    hybrid : bool  – use hybrid scoring (default True)
    """
    matches = new_df[new_df['title'] == movie]
    if matches.empty:
        print(f"Movie '{movie}' not found in dataset.")
        return

    idx  = matches.index[0]
    sim  = similarity[idx]          # cosine similarity vector

    if hybrid:
        scores = sim * 0.70 + vote_arr * 0.20 + pop_arr * 0.10
    else:
        scores = sim

    # Exclude the query movie itself, sort descending
    ranked = sorted(
        [(i, s) for i, s in enumerate(scores) if i != idx],
        key=lambda x: x[1], reverse=True
    )[:top_n]

    mode_label = '(Hybrid)' if hybrid else '(Cosine only)'
    print(f"\n🎬 Recommendations for '{movie}' {mode_label}:")
    print("-" * 55)
    for rank, (i, score) in enumerate(ranked, 1):
        row = new_df.iloc[i]
        print(f"{rank}. {row.title:<40} score={score:.4f}")
        print(f"   Poster: {row.poster}")

In [22]:
# --- Test the recommender ---
recommend('Avatar')
print()
recommend('The Dark Knight')
print()
recommend('Interstellar')


🎬 Recommendations for 'Avatar' (Hybrid):
-------------------------------------------------------
1. Aliens                                   score=0.3501
   Poster: https://image.tmdb.org/t/p/original/r1x5JGpyqZU8PYhbs4UcrO1Xb6x.jpg
2. Interstellar                             score=0.3263
   Poster: https://image.tmdb.org/t/p/original/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg
3. Guardians of the Galaxy                  score=0.3075
   Poster: https://image.tmdb.org/t/p/original/r7vmZjiyZw9rpJMQJdXpjgiCOk9.jpg
4. Star Trek Into Darkness                  score=0.2925
   Poster: https://image.tmdb.org/t/p/original/7XrRkhMa9lQ71RszzSyVrJVvhyS.jpg
5. The Book of Life                         score=0.2789
   Poster: https://image.tmdb.org/t/p/original/aotTZos5KswgCryEzx2rlOjFsm1.jpg


🎬 Recommendations for 'The Dark Knight' (Hybrid):
-------------------------------------------------------
1. The Dark Knight Rises                    score=0.5039
   Poster: https://image.tmdb.org/t/p/original/85cWkCVfti

## 8. Evaluation — Precision@K

**Precision@K** measures what fraction of the top-K recommendations are relevant.

Here we define *relevance* by matching genres — movies that share at least one
genre with the query movie are considered relevant.

In [23]:
# Re-attach genres to new_df for evaluation
new_df['genres_list'] = movies['genres'].values

def precision_at_k(movie, k=5, hybrid=True):
    """
    Compute Precision@K using genre overlap as the relevance signal.
    """
    matches = new_df[new_df['title'] == movie]
    if matches.empty:
        return None

    idx    = matches.index[0]
    sim    = similarity[idx]
    query_genres = set(new_df.iloc[idx]['genres_list'])

    scores = sim * 0.70 + vote_arr * 0.20 + pop_arr * 0.10 if hybrid else sim
    ranked = sorted(
        [(i, s) for i, s in enumerate(scores) if i != idx],
        key=lambda x: x[1], reverse=True
    )[:k]

    hits = sum(
        1 for i, _ in ranked
        if set(new_df.iloc[i]['genres_list']) & query_genres
    )
    return hits / k

# Evaluate on a few movies
test_movies = ['Avatar', 'The Dark Knight', 'Interstellar', 'Toy Story']

print(f"{'Movie':<35} {'P@5 Hybrid':>12} {'P@5 Cosine':>12}")
print("-" * 62)
for m in test_movies:
    p_hybrid = precision_at_k(m, k=5, hybrid=True)
    p_cosine = precision_at_k(m, k=5, hybrid=False)
    if p_hybrid is not None:
        print(f"{m:<35} {p_hybrid:>12.2f} {p_cosine:>12.2f}")

Movie                                 P@5 Hybrid   P@5 Cosine
--------------------------------------------------------------
Avatar                                      1.00         1.00
The Dark Knight                             1.00         1.00
Interstellar                                1.00         1.00
Toy Story                                   0.60         0.80


## 9. Saving the Model

We persist the cleaned dataframe (with poster URLs and normalized scores) and the similarity
matrix so they can be loaded instantly in a web app or API.

In [24]:
pickle.dump(new_df,     open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl',  'wb'))

print("model artifacts saved:")
print("   movie_list.pkl  — processed movie dataframe")
print("   similarity.pkl  — cosine similarity matrix")

model artifacts saved:
   movie_list.pkl  — processed movie dataframe
   similarity.pkl  — cosine similarity matrix
